# 🚀 Restaurant Reputation Intelligence System (RRIS) - Full Pipeline

สมุดโน้ตบุ๊กนี้จะพารันไปป์ไลน์ของระบบ **RRIS** ทั้งหมดตั้งแต่เริ่มต้นจนจบกระบวนการ โดยเน้นการใช้ข้อมูล **`merged_reduce2.csv`** สำหรับฝึกสอน และ **`70k_reduce.csv`** สำหรับการประเมิน (Evaluate & Score)

**ข้อควรระวัง:** สมุดโน้ตบุ๊กนี้ออกแบบมาให้รันจากโฟลเดอร์ `notebooks/` ของโปรเจกต์

## 1. Setup & Environment Check
ตั้งค่า Path ของโปรเจกต์เพื่อให้สามารถเรียกใช้โมดูล `rris` ได้อย่างถูกต้อง

In [2]:
import sys
import os
import json
import pandas as pd
from IPython.display import display, JSON, HTML

# เพิ่ม root directory เข้าไปใน python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from rris import config
print(f"✅ Project Root: {config.ROOT}")
print(f"✅ Target Training Data: {config.RAW_DATA_PATH}")
print(f"✅ Target Eval/Score Data: {config.TEST_PATH}")

✅ Project Root: F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System
✅ Target Training Data: F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\merge\merged_reduce2.csv
✅ Target Eval/Score Data: F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\70k\70k_reduce.csv


## 2. Data Preparation (ตรวจสอบข้อมูล)
ตรวจสอบว่ามีไฟล์ Train และ Test พร้อมสำหรับรันหรือไม่

In [3]:
print(f"🔍 Checking for {config.RAW_DATA_PATH} ...")
if os.path.exists(config.RAW_DATA_PATH):
    print("✅ Found merged_reduce2.csv! Ready for training.")
else:
    print("❌ merged_reduce2.csv not found! Please run the merge scripts.")

print(f"\n🔍 Checking for {config.TEST_PATH} ...")
if os.path.exists(config.TEST_PATH):
    print("✅ Found 70k_reduce.csv! Ready for evaluation/scoring.")
else:
    print("❌ 70k_reduce.csv not found!")

🔍 Checking for F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\merge\merged_reduce2.csv ...
✅ Found merged_reduce2.csv! Ready for training.

🔍 Checking for F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\70k\70k_reduce.csv ...
✅ Found 70k_reduce.csv! Ready for evaluation/scoring.


## 3. Train Models (ฝึกสอนโมเดล)
ฝึกสอนโมเดลหลัก **Baseline** และ **Embedding** ด้วยข้อมูล Train (`merged_reduce2.csv`)

In [4]:
print("🚀 Training Baseline Model...")
!{sys.executable} -m rris train baseline

🚀 Training Baseline Model...
--- Step 1: Loading data ---
Cleaning stats (train_reduce):
  initial rows:      6990
  removed empty:     0
  removed short:     90
  removed duplicate: 61
  final rows:        6839
Saved holdout split: F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\merge\holdout.csv (n=1368)

--- Rating distribution: train split (after clean) (n=4376) ---
  star 1: 296 (6.8%)
  star 2: 196 (4.5%)
  star 3: 825 (18.9%)
  star 4: 1581 (36.1%)
  star 5: 1478 (33.8%)

--- Rating distribution: HF test (after clean) (n=4603) ---
  star 1: 191 (4.1%)
  star 2: 98 (2.1%)
  star 3: 323 (7.0%)
  star 4: 974 (21.2%)
  star 5: 3017 (65.5%)

--- Train vs test share delta (test% - train%) ---
  star 1: -2.6 pp
  star 2: -2.3 pp
  star 3: -11.8 pp
  star 4: -15.0 pp
  star 5: +31.8 pp
Mock mix (fraction=0.2): 4376 -> 5251 rows
Undersample 4-star (keep=0.65): 5251 -> 4613 rows
Oversampled low stars (factor=5): 4613 -> 7341 rows

--- Data Augmentation (minority cl

In [5]:
print("🚀 Training Embedding Model...")
!{sys.executable} -m rris train embedding

🚀 Training Embedding Model...
--- Step 1: Loading data ---
Cleaning stats (embedding_train_pool):
  initial rows:      6990
  removed empty:     0
  removed short:     92
  removed duplicate: 61
  final rows:        6837

--- Step 2: Extracting Sentence Embeddings ---
Loading embedding model: intfloat/multilingual-e5-base
Encoding 5469 training texts...
Done encoding train in 25.45s
Encoding 1368 validation texts...
Caching embeddings...

--- Step 3: Training and Comparing Classifiers ---

[2/4] Training XGBoost...

--- Classifier Comparative Analysis ---
Classifier                | Val Acc    | Val MAE    | Val F1-Macro
-----------------------------------------------------------------
xgboost                   | 0.5183     | 0.5850     | 0.4886      
-----------------------------------------------------------------

WINNER: XGBOOST (Val MAE: 0.5850, F1-Macro: 0.4886)

--- Step 4: Saving artifacts ---
Saved XGBOOST classifier to F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligen


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2575.73it/s]

Batches: 100%|██████████| 171/171 [00:25<00:00,  6.73it/s]

Batches: 100%|██████████| 43/43 [00:06<00:00,  6.71it/s]
f:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\.venv\lib\site-packages\xgboost\core.py:751: UserWarning: [06:54:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [6]:
print("🚀 Training XLM-R Model...")
!{sys.executable} -m rris train xlmr

🚀 Training XLM-R Model...
PyTorch 2.6.0+cu124 | TORCH_DEVICE=cuda
Using GPU: NVIDIA GeForce RTX 2060 (cuda:0)
--- Step 1: Initialize Tokenizer and Model (device=cuda, mode=Ordinal Regression (1.0 to 5.0)) ---
  > Using preprocessing strategy: keep_digits
Cleaning stats (xlmr_train_pool):
  initial rows:      6990
  removed empty:     0
  removed short:     92
  removed duplicate: 61
  final rows:        6837

--- Data Augmentation (minority classes) ---
  1 stars: 370 -> augmenting 2130 more
  2 stars: 245 -> augmenting 2255 more
  3 stars: 1032 -> augmenting 1468 more

  Augmentation: 5469 -> 11322 rows (+5853)

  Label mapping: Ordinal Regression (1.0 to 5.0)
Rating distribution (train):
  star 1: 2500
  star 2: 2500
  star 3: 2500
  star 4: 1976
  star 5: 1846
  > Using MSELoss for Ordinal Regression
--- Step 2: Training (batch=24, accum=1, effective_batch=24, max_length=128, amp=True) ---
  train batch 47/472 loss=3.7242
  train batch 94/472 loss=0.8203
  train batch 141/472 loss=0

F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\src\rris\config.py:87: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  x = torch.randn(8, 8, device="cuda")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2954.46it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.den

## 4. Evaluation (การประเมินประสิทธิภาพ)
ระบบจะทำการประเมินบน `70k_reduce.csv` โดยอัตโนมัติตามที่กำหนดไว้ใน config `TEST_PATH`

In [7]:
print(f"📊 Evaluating all models on {config.TEST_PATH}...")
!{sys.executable} -m rris evaluate --model all --output ../outputs/eval/eval_report.json

📊 Evaluating all models on F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\70k\70k_reduce.csv...
--- Evaluation (model=all, active=['baseline', 'embedding', 'xlmr']) ---
Input: F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\data\70k\70k_reduce.csv
Cleaning stats (70k_reduce.csv):
  initial rows:      5000
  removed empty:     0
  removed short:     55
  removed duplicate: 342
  final rows:        4603

--- Rating distribution: eval input (baseline clean) (n=4603) ---
  star 1: 191 (4.1%)
  star 2: 98 (2.1%)
  star 3: 323 (7.0%)
  star 4: 974 (21.2%)
  star 5: 3017 (65.5%)

=== Majority baseline (predict 4 stars) ===
n_samples:   4603
MAE:         0.8927
RMSE:        1.0882
Accuracy:    0.2116
F1 macro:    0.0699
F1 weighted: 0.0739
Per-class recall:
  star 1: 0.0000
  star 2: 0.0000
  star 3: 0.0000
  star 4: 1.0000
  star 5: 0.0000

Classification report:
              precision    recall  f1-score   support

           1       0.00      0.0


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4612.61it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2360.99it/s]

Batches: 100%|██████████| 144/144 [00:08<00:00, 17.63it/s]
f:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\.venv\lib\site-packages\xgboost\core.py:751: UserWarning: [07:29:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [8]:
# แสดงผลลัพธ์จาก Evaluation Report
report_path = os.path.join(config.ROOT, "outputs", "eval", "eval_report.json")
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        report = json.load(f)
    
    print("🏆 Best Model:", report.get("best_model"))
    display(JSON(report))
else:
    print("❌ Evaluation report not found.")

🏆 Best Model: xlmr


<IPython.core.display.JSON object>

## 5. Scoring & Anomaly Detection (อนุมานผลและค้นหาความผิดปกติ)
เราจะใช้ `70k_reduce.csv` มา Score เพื่อหา Anomaly (สังเกตว่ามีการระบุ `--input ../data/70k/70k_reduce.csv` เพื่อความชัดเจน)

In [9]:
print("🔍 Scoring 70k_reduce test data with the baseline model...")
!{sys.executable} -m rris score --model baseline --input ../data/70k/70k_reduce.csv --output ../outputs/scores/scored_baseline.csv

scored_df = pd.read_csv(os.path.join(config.ROOT, "outputs", "scores", "scored_baseline.csv"))
display(scored_df.head(5))

🔍 Scoring 70k_reduce test data with the baseline model...
--- Running Inference & Integrity Check (model=baseline) ---
Cleaning stats (70k_reduce.csv):
  initial rows:      5000
  removed empty:     0
  removed short:     55
  removed duplicate: 342
  final rows:        4603
--- Running Aspect-Based Sentiment Analysis (ABSA) ---
Finished scoring! Check result at '../outputs/scores/scored_baseline.csv'


,text,user_rating,place_name,ai_expected_rating,aspect_food,aspect_service,aspect_atmosphere,ai_hex_color,delta,is_anomaly
0,มาลองกินตี๋น้อยสาขานี้ครับ กว้างดี อาหารมาไว พ...,5,สุกี้ตี๋น้อย พระราม2,3.686092,3.321456,2.589096,3.484576,#4caf50,1.313908,False
1,"good place, delicious food, reasonable price, ...",4,ครัวเจ๊ง้อ พระราม2,4.137990,4.137990,4.137990,4.137990,#4caf50,0.137990,False
2,the food is delicious. came here for the 0th t...,5,อิ่มภิรมย์ (IMPHIROM),4.781827,4.781827,4.781827,4.781827,#00bcd4,0.218173,False
3,so delicious that you have to repeat it. the d...,5,กฤษฎาสเต็กเฮาส์,3.391736,3.391736,3.391736,3.391736,#fbc02d,1.608264,False
4,คั่วไก่อร่อย ราคาไม่แพง คุ้มค่า แนะนำเลยครับ,5,คั่วไก่เจ๊เนี้ยว2524,4.796911,2.880158,4.796911,4.796911,#00bcd4,0.203089,False


## 6. Inference in Python API (การใช้งานในโค้ด Python)
การเรียกใช้งาน Model ด้วยโค้ด Python เพื่อรองรับการนำไปต่อยอด

In [10]:
from rris.inference.baseline import predict_baseline
from rris.inference.embedding import predict_embedding

# ข้อมูลทดสอบ 3 รีวิว (ด้านบวก ด้านลบ และปานกลาง)
sample_texts = pd.DataFrame({
    "text": [
        "อาหารอร่อยมาก บริการดีเยี่ยม คุ้มราคาที่สุด ไปซ้ำแน่นอน",
        "รสชาติเฉยๆ พนักงานบริการไม่ค่อยดี รอนานมาก ไม่แนะนำเลย",
        "พอใช้ได้ บรรยากาศร้านโอเค แต่ราคาแพงไปนิดเมื่อเทียบกับปริมาณ"
    ]
})

try:
    sample_texts["baseline_pred"] = predict_baseline(sample_texts)
    sample_texts["embedding_pred"] = predict_embedding(sample_texts)
    display(sample_texts)
except Exception as e:
    print(f"Inference failed: {e}\nPlease ensure models are trained successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

f:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\.venv\lib\site-packages\xgboost\core.py:751: UserWarning: [07:30:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,text,baseline_pred,embedding_pred
0,อาหารอร่อยมาก บริการดีเยี่ยม คุ้มราคาที่สุด ไป...,4.966575,4.855560
1,รสชาติเฉยๆ พนักงานบริการไม่ค่อยดี รอนานมาก ไม่...,1.743727,2.173581
2,พอใช้ได้ บรรยากาศร้านโอเค แต่ราคาแพงไปนิดเมื่อ...,2.285906,2.657723


## 7. Web App Preparation (การเตรียมข้อมูลสำหรับ Web Dashboard)
รันสคริปต์เตรียมข้อมูล JSON ให้กับ Frontend โดยมันจะเลือก Best Model อัตโนมัติ (`--model auto`)

In [11]:
print("🌐 Initializing Web Data...")
!{sys.executable} ../scripts/initialize_web_data.py --model auto

🌐 Initializing Web Data...
Python data initializer running...
Auto-selected model: xlmr
Cleaning stats (70k.tsv):
  initial rows:      70973
  removed empty:     0
  removed short:     6386
  removed duplicate: 22236
  final rows:        42351
Predicting with model: xlmr
Scoring complete! Saved to F:\ComSci\Coding\Project\Restaurant-Reputation-Intelligence-System\web_app\scored_reviews.json



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6246.15it/s]


### 🎉 ไปป์ไลน์สมบูรณ์แล้ว!
ตอนนี้โปรเจกต์พร้อมสำหรับการเปิด Web Application แล้ว

เนื่องจากการรัน Server เป็นโปรเซสที่ไม่สิ้นสุด แนะนำให้**เปิด Terminal แยก**ขึ้นมาที่โฟลเดอร์ root ของโปรเจกต์และพิมพ์คำสั่งต่อไปนี้:

```bash
cd web_app
bun install
cd ..
bun run web_app/index.ts
```
จากนั้นเข้าไปที่ [http://127.0.0.1:8000](http://127.0.0.1:8000) เพื่อดูผลลัพธ์!